In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold

from skopt import BayesSearchCV
from skopt.space import Real, Integer

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
import warnings, os, joblib
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import train_test_split

####  Load Training and Testing Dataset

In [2]:
df = pd.read_csv("../dataset/Clean_CICDDOS_Final_Dataset.csv")

print("Train Shape:", df.shape)
# Train Shape: (461158, 67)

Train Shape: (346873, 67)


In [3]:
df.columns

Index(['Protocol', 'Flow_Duration', 'Total_Fwd_Packets',
       'Total_Backward_Packets', 'Total_Length_of_Fwd_Packets',
       'Total_Length_of_Bwd_Packets', 'Fwd_Packet_Length_Max',
       'Fwd_Packet_Length_Min', 'Fwd_Packet_Length_Mean',
       'Fwd_Packet_Length_Std', 'Bwd_Packet_Length_Max',
       'Bwd_Packet_Length_Min', 'Bwd_Packet_Length_Mean',
       'Bwd_Packet_Length_Std', 'Flow_Bytes/s', 'Flow_Packets/s',
       'Flow_IAT_Mean', 'Flow_IAT_Std', 'Flow_IAT_Max', 'Flow_IAT_Min',
       'Fwd_IAT_Total', 'Fwd_IAT_Mean', 'Fwd_IAT_Std', 'Fwd_IAT_Max',
       'Fwd_IAT_Min', 'Bwd_IAT_Total', 'Bwd_IAT_Mean', 'Bwd_IAT_Std',
       'Bwd_IAT_Max', 'Bwd_IAT_Min', 'Fwd_PSH_Flags', 'Fwd_Header_Length',
       'Bwd_Header_Length', 'Fwd_Packets/s', 'Bwd_Packets/s',
       'Min_Packet_Length', 'Max_Packet_Length', 'Packet_Length_Mean',
       'Packet_Length_Std', 'Packet_Length_Variance', 'SYN_Flag_Count',
       'RST_Flag_Count', 'ACK_Flag_Count', 'URG_Flag_Count', 'CWE_Flag_Count',
      

In [4]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Protocol,346873.0,1.592489e+01,3.292063e+00,0.0,17.0,17.0,17.0,1.700000e+01
Flow_Duration,346873.0,1.191548e+06,9.023881e+06,0.0,1.0,2.0,1106.0,1.200000e+08
Total_Fwd_Packets,346873.0,1.006427e+01,3.679485e+02,1.0,2.0,2.0,4.0,9.953800e+04
Total_Backward_Packets,346873.0,7.561240e-01,1.768528e+01,0.0,0.0,0.0,0.0,4.632000e+03
Total_Length_of_Fwd_Packets,346873.0,4.008724e+03,3.359700e+04,0.0,458.0,1228.0,2944.0,1.526642e+07
...,...,...,...,...,...,...,...,...
Idle_Std,346873.0,3.202217e+04,6.354723e+05,0.0,0.0,0.0,0.0,6.188006e+07
Idle_Max,346873.0,4.320246e+05,4.071837e+06,0.0,0.0,0.0,0.0,1.192194e+08
Idle_Min,346873.0,3.791161e+05,3.748339e+06,0.0,0.0,0.0,0.0,1.192194e+08
Inbound,346873.0,9.253877e-01,2.627651e-01,0.0,1.0,1.0,1.0,1.000000e+00


In [5]:
df['Label'].unique()

array([ 3,  5,  7,  0,  4, 10,  6,  1,  2, 11,  8,  9])

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 346873 entries, 0 to 346872
Data columns (total 67 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Protocol                     346873 non-null  int64  
 1   Flow_Duration                346873 non-null  int64  
 2   Total_Fwd_Packets            346873 non-null  int64  
 3   Total_Backward_Packets       346873 non-null  int64  
 4   Total_Length_of_Fwd_Packets  346873 non-null  float64
 5   Total_Length_of_Bwd_Packets  346873 non-null  float64
 6   Fwd_Packet_Length_Max        346873 non-null  float64
 7   Fwd_Packet_Length_Min        346873 non-null  float64
 8   Fwd_Packet_Length_Mean       346873 non-null  float64
 9   Fwd_Packet_Length_Std        346873 non-null  float64
 10  Bwd_Packet_Length_Max        346873 non-null  float64
 11  Bwd_Packet_Length_Min        346873 non-null  float64
 12  Bwd_Packet_Length_Mean       346873 non-null  float64
 13 

#### Train Test split

- separate features and labels

In [7]:
X = df.drop('Label', axis=1)
y = df['Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y  # ← critical
)
X_train.shape, X_test.shape

((277498, 66), (69375, 66))

#### Strong feature stabilization

In [8]:
# replace inf
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

X_train.isnull().sum().sum(), X_test.isnull().sum().sum()

(np.int64(0), np.int64(0))

In [ ]:
# X_train = X_train.clip(0, 1e5)
# X_test = X_test.clip(0, 1e5)

# X_train.isnull().sum().sum(), X_test.isnull().sum().sum()

(np.int64(0), np.int64(0))

#### Handle the class imbalance in the Model

In [9]:
# calculate wights to balance the 13 classes
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

#### Gradient Boosting Model

In [ ]:
# gb_model = GradientBoostingClassifier(
#     n_estimators=150, # number of trees (More tree better model but slow)
#     learning_rate=0.05, # shrinkage factor per tree
#     max_depth=4,        # max depth of each individual tree
#     subsample=0.8,     # use 80% data per tree (adds variance diversity)
#     min_samples_leaf=20,  # min samples required in a leaf
#     max_features='sqrt',      # consider sqrt(n_features) at each split → faster
#     validation_fraction=0.1,
#     n_iter_no_change=10, # early stopping
#     # tol=0.01,  # stop if loss doesn't imporve by this much
#     random_state=42,
#     verbose=2          # print progress
# )

In [10]:
gb_model = GradientBoostingClassifier(
    n_estimators=200,     
    learning_rate=0.03,   
    max_depth=10,         
    subsample=0.8,
    max_features='sqrt',
    validation_fraction=0.1,
    n_iter_no_change=15,
    random_state=42,
    verbose=2
)

In [11]:
print("Training Model ...")
gb_model.fit(X_train, y_train, sample_weight=sample_weights)
print("Model Training Completed")

Training Model ...
      Iter       Train Loss      OOB Improve   Remaining Time 
         1           2.2650           0.2187           48.15m
         2           2.0971           0.1677           69.10m
         3           1.9602           0.1371           76.13m
         4           1.8439           0.1164           79.92m
         5           1.7428           0.0994           82.81m
         6           1.6540           0.0874           83.30m
         7           1.5753           0.0804           84.57m
         8           1.5047           0.0732           84.73m
         9           1.4394           0.0601           85.64m
        10           1.3817           0.0633           85.71m
        11           1.3270           0.0514           86.18m
        12           1.2782           0.0541           85.95m
        13           1.2308           0.0409           85.78m
        14           1.1881           0.0430           84.74m
        15           1.1489           0.0401      

#### Model Evaluation

In [12]:
y_pred = gb_model.predict(X_test)

In [13]:
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.7728576576576577


- 1st model : Accuracy: 0.7507312567352594
- 2nd model : Accuracy: 0.7640093395597065
- 3rd model : Accuracy: 0.7728576576576577



In [14]:
label_encoder = joblib.load('../models/LabelEncoder.pkl')
class_names = list(label_encoder.classes_)
print("\nClassification Report")
print(classification_report(y_test, y_pred, target_names=class_names))


Classification Report
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00      6000
         DNS       0.75      0.51      0.61      6000
        LDAP       0.61      0.68      0.65      6000
       MSSQL       0.93      0.95      0.94      6000
         NTP       0.99      1.00      1.00      6000
     NetBIOS       0.76      0.40      0.52      6000
     Portmap       0.60      0.91      0.72      6000
        SNMP       0.71      0.84      0.77      6000
        SSDP       0.56      0.70      0.62      6000
        TFTP       1.00      0.99      0.99      6000
         UDP       0.61      0.46      0.52      6000
      UDPLag       0.95      0.91      0.93      3375

    accuracy                           0.77     69375
   macro avg       0.79      0.78      0.77     69375
weighted avg       0.78      0.77      0.77     69375



In [15]:

print("\nConfusion Matrix")
confusion_matrix(y_test, y_pred)


Confusion Matrix


array([[5998,    2,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0],
       [   3, 3054, 2048,  151,    8,    2,    3,  702,   17,    0,    7,
           5],
       [   0,  742, 4101,    9,    0,    0,    0, 1148,    0,    0,    0,
           0],
       [   1,   47,   63, 5699,   20,    1,    1,   96,   45,    7,   16,
           4],
       [   4,    2,    0,    4, 5984,    0,    0,    0,    1,    4,    0,
           1],
       [   1,    2,    1,   42,    0, 2373, 3574,    3,    0,    0,    2,
           2],
       [   2,    1,    1,    1,    0,  558, 5431,    0,    0,    0,    0,
           6],
       [   0,  185,  487,   35,    0,  175,   63, 5029,    8,    0,    0,
          18],
       [   1,    6,    3,   98,    0,    2,    0,   67, 4214,    0, 1576,
          33],
       [   1,    1,    1,    0,    4,    0,    1,    1,    0, 5933,    0,
          58],
       [   1,    3,    2,   73,    0,    2,    3,    0, 3147,    0, 2735,
          34],
       [   0,    1,  

In [16]:
fig, ax = plt.subplots(figsize=(12, 10))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, colorbar=True, xticks_rotation=45)
ax.set_title("Gradient Boost — Confusion Matrix (CIC-DDoS 2019)", fontsize=13)
plt.tight_layout()
cm_path = os.path.join('../results/confusion_matrix4.png')
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"\n  ✓ Confusion matrix saved → {cm_path}")


  ✓ Confusion matrix saved → ../results/confusion_matrix4.png
